# 0. INTRO

- Orchestrator-worker란?
    - 주어진 명령에 따라 동적으로 서브 에이전트를 Tool로 활용하고, 그 결과물을 병합하여 최종 전달하는 방식
    - `Send API`로 동적인 병렬 sub-agent 실행 가능
- Supervisor, Sub-agent

# 1. API 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

### LangSmith: trace 역할

In [ ]:
# print(os.getenv('LANGSMITH_API_KEY'))
# print(os.getenv('LANGSMITH_TRACING'))
# print(os.getenv('LANGSMITH_PROJECT'))

# 2. Low-level 도구(Tool) 정의

| 도구 | API | 역할 |
|------|-----|------|
| `search_web` | EXA | 웹 검색 (논문, 기사, 문서) |
| `search_news` | EXA | 최신 뉴스 검색 (날짜 필터) |
| `validate_markdown_section` | 로컬 | 마크다운 품질 검증(3000자면 진짜 했는지) |
| `e2b_code_interpreter` | E2B | 격리된 샌드박스에서 Python 코드 실행 |

- search_web

In [ ]:
from exa_py import Exa
from langchain.tools import tool
from datetime import datetime, timedelta

# EXA 클라이언트 초기화
exa_client = Exa(api_key=os.environ.get("EXA_API_KEY"))

@tool
def search_web(query: str, num_results: int = 5) -> str:
    """EXA API를 사용하여 웹 검색을 수행합니다.

    최신 연구 동향, 기술 정보, 논문, 문서 등을 검색할 때 사용하세요.

    Args:
        query: 검색할 쿼리
        num_results: 반환할 결과 수 (기본값: 5)
    """
    try:
        results = exa_client.search_and_contents(
            query=query,
            type="auto",
            num_results=num_results,
            text={"max_characters": 2000},
        )

        if not results.results:
            return f"'{query}'에 대한 검색 결과가 없습니다."

        output = []
        for i, result in enumerate(results.results, 1):
            output.append(f"{i}. **{result.title}**")
            output.append(f"   URL: {result.url}")
            if result.text:
                snippet = result.text[:500] + "..." if len(result.text) > 500 else result.text
                output.append(f"   {snippet}")
            output.append("")

        return "\n".join(output)

    except Exception as e:
        return f"검색 오류: {str(e)}"

- search_news

In [ ]:
@tool
def search_news(query: str, days: int = 7) -> str:
    """EXA API를 사용하여 최신 뉴스를 검색합니다.

    최근 뉴스, 트렌드, 업계 동향을 파악할 때 사용하세요.

    Args:
        query: 검색할 뉴스 주제
        days: 검색할 기간 (기본값: 최근 7일)
    """
    try:
        start_date = (datetime.now() - timedelta(days=days)).strftime("%Y-%m-%d")

        results = exa_client.search_and_contents(
            query=query,
            type="auto",
            num_results=5,
            start_published_date=start_date,
            text={"max_characters": 2000},
        )

        if not results.results:
            return f"최근 {days}일 내 '{query}' 관련 뉴스가 없습니다."

        output = [f"최근 {days}일 뉴스 ('{query}'):\n"]
        for i, result in enumerate(results.results, 1):
            output.append(f"{i}. **{result.title}**")
            output.append(f"   URL: {result.url}")
            if hasattr(result, 'published_date') and result.published_date:
                output.append(f"   발행일: {result.published_date}")
            if result.text:
                snippet = result.text[:500] + "..." if len(result.text) > 500 else result.text
                output.append(f"   {snippet}")
            output.append("")

        return "\n".join(output)

    except Exception as e:
        return f"뉴스 검색 오류: {str(e)}"

- validate_markdown_section

In [ ]:
import re
from e2b_code_interpreter import Sandbox


@tool
def validate_markdown_section(
    section_name: str,
    markdown: str,
    min_words: int = 50,
    max_words: int = 300,
) -> dict:
    """작성된 섹션의 마크다운 품질을 검증합니다.

    Returns:
        {"ok": bool, "word_count": int, "issues": list[str]}
    """
    issues: list[str] = []
    words = re.findall(r"\b\w+\b", markdown)
    wc = len(words)

    if wc < min_words:
        issues.append(f"너무 짧음: {wc} < {min_words} 단어")
    if wc > max_words:
        issues.append(f"너무 김: {wc} > {max_words} 단어")

    first_lines = "\n".join(markdown.splitlines()[:3]).strip()
    if "##" not in first_lines:
        issues.append("마크다운 제목(예: '## ...')이 상단에 없음")

    return {"ok": len(issues) == 0, "word_count": wc, "issues": issues}

- e2b_code_interpreter

In [ ]:
# ── E2B 샌드박스 생성 ──
# timeout: 샌드박스 유지 시간 (초). 기본값 300초(5분)이므로
# 노트북 실습 중 만료되지 않도록 넉넉히 설정합니다.
# 만료 시 셀을 다시 실행하면 새 샌드박스가 생성됩니다.
sandbox = Sandbox.create(timeout=600)  # 10분


@tool
def e2b_code_interpreter(code: str) -> str:
    """E2B 클라우드 샌드박스에서 Python 코드를 실행합니다.

    데이터 분석, 계산, 시각화 등 코드 실행이 필요할 때 사용하세요.
    격리된 환경에서 안전하게 실행되며, matplotlib 등 라이브러리 사용 가능합니다.

    Args:
        code: 실행할 Python 코드
    """
    execution = sandbox.run_code(code)

    result_parts = []
    if execution.text:
        result_parts.append(execution.text)
    if execution.logs.stdout:
        stdout = "".join(execution.logs.stdout)
        if stdout.strip():
            result_parts.append(stdout.strip())
    if execution.logs.stderr:
        stderr = "".join(execution.logs.stderr)
        if stderr.strip():
            result_parts.append(f"[stderr] {stderr.strip()}")
    if execution.error:
        result_parts.append(f"Error: {execution.error.name}: {execution.error.value}")

    return "\n".join(result_parts) if result_parts else "코드가 성공적으로 실행되었습니다."

# 3. 전문 Sub-agent 생성 

| Sub-agent | 도구 | 역할 |
|-----------|------|------|
| Research Agent | `search_web`, `search_news` | EXA로 웹/뉴스 검색 |
| Writer Agent | `validate_markdown_section` | 마크다운 섹션 작성 + 검증 |
| Code Agent | `e2b_code_interpreter` | E2B 샌드박스에서 코드 실행 |

In [ ]:
from datetime import datetime

today = datetime.today().strftime('%Y-%m-%d')
date_prompt=f"""
!!!!! 반드시 주의 !!!!!
[중요] 오늘 날짜는 반드시 {today}입니다. 
절대로 이전 연도(2023/2024 등)가 아닌, {today} 기준으로만 분석하세요.
날짜를 임의로 유추해서 쓰지 말 것.\n"""

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

- research agent: `search_web`, `search_news`

In [ ]:
# =====================
# 1) Research Agent: EXA 웹 검색 전문
# =====================

RESEARCH_AGENT_PROMPT = (
    "당신은 기술 리서치 전문 에이전트입니다. "
    "주어진 주제에 대해 search_web으로 관련 논문/기사/문서를 검색하고, "
    "필요하면 search_news로 최신 동향도 파악하세요. "
    "검색 결과를 종합하여 핵심 발견사항을 정리해서 응답하세요. "
    "반드시 출처(URL)와 연도를 포함하세요."
)

research_agent = create_agent(
    llm,
    tools=[search_web, search_news],
    system_prompt=date_prompt+RESEARCH_AGENT_PROMPT,
)

- write agent: `validate_markdown_section`

In [ ]:
# =====================
# 2) Writer Agent: 섹션 작성 전문
# =====================

WRITER_AGENT_PROMPT = (
    "당신은 기술 보고서 섹션 작성 전문 에이전트입니다. "
    "주어진 리서치 결과를 바탕으로 마크다운 형식의 섹션을 작성하세요. "
    "작성 후 반드시 validate_markdown_section을 호출하여 품질을 검증하세요. "
    "검증에 실패하면 수정 후 다시 검증하세요. "
    "최종 응답에는 완성된 마크다운 섹션만 포함하세요."
)

writer_agent = create_agent(
    llm,
    tools=[validate_markdown_section],
    system_prompt=WRITER_AGENT_PROMPT,
)

- Code Agent:  `e2b_code_interpreter`

In [ ]:
# =====================
# 3) Code Agent: E2B 샌드박스 코드 실행 전문
# =====================

CODE_AGENT_PROMPT = (
    "당신은 데이터 분석 및 코드 실행 전문 에이전트입니다. "
    "e2b_code_interpreter를 사용하여 E2B 클라우드 샌드박스에서 Python 코드를 실행하세요. "
    "데이터 분석, 수학 계산, 시각화, 통계 등의 작업을 수행합니다. "
    "코드에서는 print()로 결과를 출력하세요. "
    "에러가 발생하면 코드를 수정하여 다시 실행하세요. "
    "최종 응답에는 분석 결과와 인사이트를 정리해서 포함하세요."
)

code_agent = create_agent(
    llm,
    tools=[e2b_code_interpreter],
    system_prompt=CODE_AGENT_PROMPT,
)

## 3-(1) Sub-agent 개별 테스트

In [ ]:
# Research Agent 테스트 (EXA 실시간 웹 검색)
query = "LLM Scaling Laws에 대한 최신 연구 동향을 조사해주세요"
print(query)
print("=" * 60)
 
for step in research_agent.stream( # research_agent
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:
# Code Agent 테스트 (E2B 샌드박스에서 실행)
query = (
    "1부터 100까지 소수를 구하고, "
    "소수의 분포를 10단위 구간별로 분석해주세요. "
    "결과를 표 형태로 보여주세요."
)

print(query)
print("=" * 60)

for step in code_agent.stream( # code_agent
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

# 4. Supervisor
- `research_topic`, `write_section`, `run_code`

In [ ]:
@tool
def research_topic(request: str) -> str:
    """주제를 리서치하여 핵심 발견사항을 정리합니다.

    기술 동향, 연구 결과, 최신 뉴스가 필요할 때 사용하세요.
    Sub-agent가 EXA API로 웹 검색 및 뉴스 검색을 자동으로 수행합니다.

    Input: 자연어 리서치 요청 (e.g., 'Transformer 아키텍처의 발전 과정 조사')
    """
    result = research_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    # Sub-agent의 최종 응답만 Supervisor에게 반환 (중간 추론/도구 호출은 숨김)
    return result["messages"][-1].content

In [ ]:
@tool
def write_section(request: str) -> str:
    """리서치 결과를 바탕으로 보고서 섹션을 작성합니다.

    마크다운 형식의 섹션 작성이 필요할 때 사용하세요.
    Sub-agent가 작성 및 품질 검증을 자동으로 수행합니다.

    Input: 작성할 섹션 정보 (제목, 리서치 결과, 요구사항 포함)
    """
    result = writer_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].content

In [ ]:
@tool
def run_code(request: str) -> str:
    """데이터 분석이나 계산을 위해 Python 코드를 실행합니다.

    수치 분석, 통계 처리, 데이터 시각화가 필요할 때 사용하세요.
    Sub-agent가 E2B 샌드박스에서 안전하게 코드를 작성하고 실행합니다.

    Input: 자연어 분석 요청 (e.g., '연도별 논문 수 추이를 분석하고 차트 생성')
    """
    result = code_agent.invoke(
        {"messages": [{"role": "user", "content": request}]}
    )
    return result["messages"][-1].content

In [ ]:
SUPERVISOR_PROMPT = (
    "당신은 기술 보고서 작성을 총괄하는 Supervisor 에이전트입니다. "
    "사용자의 요청을 분석하여 적절한 도구를 호출하세요:\n\n"
    "1. research_topic: 주제 리서치가 필요할 때 (EXA 웹 검색)\n"
    "2. write_section: 리서치 결과를 바탕으로 섹션 작성할 때\n"
    "3. run_code: 데이터 분석, 계산, 시각화가 필요할 때 (E2B 샌드박스)\n\n"
    "복합 요청의 경우, 먼저 리서치하고 필요시 코드 분석을 한 후, "
    "그 결과를 바탕으로 섹션을 작성하세요. "
    "최종 응답에서 전체 결과를 종합하여 사용자에게 보고하세요."
)

supervisor = create_agent(
    llm,
    tools=[research_topic, write_section, run_code],
    system_prompt=SUPERVISOR_PROMPT,
)

## 4-(1). 단일 도메인 요청 (리서치만)

In [ ]:
query = "최근 codex앱과 claude code에 대한 사용자들의 후기를 모아 비교한 결과를 보고서에 작성해주세요."

print(query)
print("=" * 60)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

- request : `research_topic` | codex 앱 사용자 후기 최근 동향 및 평가
- request : `research_topic` | claude code 사용자 후기 최근 동향 및 평가

In [ ]:
message

In [ ]:
print(message.content)

## 4-(2). 복합 요청 (리서치 → 분석 → 작성)

In [ ]:
sandbox = Sandbox.create(timeout=600)

query = (
    "LLM Scaling Laws에 대해 웹에서 리서치한 후, "
    "2025년과 2026년에 발표된 주요 모델들의 파라미터 수나 다양한 벤치마크 성적 등을 코드로 비교 분석하고, "
    "그 결과를 바탕으로 '2025년 이후 Scaling Laws의 현황' 보고서 섹션을 작성해주세요."
)

print(query)
print("=" * 60)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()